# Lab 7 — Bounded Reflection for an Operator Brief

**Optional · 60 minutes · Level 200**

Wrap a cyclic workflow as an agent. A generator drafts an operator brief; a
deterministic reviewer requests a revision when required fields are missing.
The loop is bounded to prevent unending model/reviewer cycles.

## Learning objectives

- Exchange typed messages between custom executors.
- Add a deterministic quality contract.
- Bound retries and expose a final result.
- Run a workflow through the familiar agent interface.

In [ ]:
import os
import re

from dotenv import find_dotenv, load_dotenv

env_path = find_dotenv(usecwd=True)
if env_path:
    load_dotenv(env_path)


def safe_name(value: str, *, max_length: int = 40) -> str:
    value = re.sub(r"[^a-z0-9-]+", "-", value.lower()).strip("-")
    value = re.sub(r"-+", "-", value)
    if not value:
        raise ValueError("Resource namespace must contain a letter or number.")
    return value[:max_length].rstrip("-")


raw_namespace = (
    os.getenv("WORKSHOP_RESOURCE_NAMESPACE")
    or os.getenv("WORKSHOP_TEAM_ID")
    or os.getenv("WORKSHOP_PARTICIPANT_ID")
)
if not raw_namespace:
    raise ValueError(
        "Set WORKSHOP_RESOURCE_NAMESPACE (preferred), WORKSHOP_TEAM_ID, "
        "or WORKSHOP_PARTICIPANT_ID before running workshop labs."
    )

RESOURCE_NAMESPACE = safe_name(raw_namespace)
PROJECT_ENDPOINT = os.environ["FOUNDRY_PROJECT_ENDPOINT"]
MODEL = os.environ["FOUNDRY_MODEL"]

print(f"Namespace: {RESOURCE_NAMESPACE}")
print(f"Model: {MODEL}")

In [ ]:
from dataclasses import dataclass
from uuid import uuid4

from agent_framework import (
    AgentResponseUpdate,
    Content,
    Executor,
    Message,
    SupportsChatGetResponse,
    WorkflowBuilder,
    WorkflowContext,
    handler,
)
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

In [ ]:
@dataclass
class ReviewRequest:
    request_id: str
    user_messages: list[Message]
    agent_messages: list[Message]
    attempt: int


@dataclass
class ReviewResponse:
    request_id: str
    approved: bool
    feedback: str
    attempt: int

In [ ]:
class DeterministicBriefReviewer(Executor):
    REQUIRED_LABELS = ("SOURCE:", "CONFIDENCE:", "NEXT ACTION:")
    UNSAFE_IMPERATIVES = (
        "open the breaker",
        "close the breaker",
        "energize the",
    )

    @handler
    async def review(
        self,
        request: ReviewRequest,
        ctx: WorkflowContext[ReviewResponse, ReviewResponse],
    ) -> None:
        text = "\n".join(message.text for message in request.agent_messages)
        missing = [
            label for label in self.REQUIRED_LABELS if label not in text.upper()
        ]
        unsafe = [
            phrase
            for phrase in self.UNSAFE_IMPERATIVES
            if phrase in text.lower()
        ]
        approved = not missing and not unsafe
        feedback_parts = []
        if missing:
            feedback_parts.append(f"Add labels: {', '.join(missing)}")
        if unsafe:
            feedback_parts.append(
                f"Remove direct operational commands: {', '.join(unsafe)}"
            )
        feedback = "; ".join(feedback_parts) or "Quality contract satisfied."
        await ctx.send_message(
            ReviewResponse(
                request_id=request.request_id,
                approved=approved,
                feedback=feedback,
                attempt=request.attempt,
            )
        )

In [ ]:
class BriefGenerator(Executor):
    MAX_ATTEMPTS = 2

    def __init__(self, id: str, chat_client: SupportsChatGetResponse) -> None:
        super().__init__(id=id)
        self._chat_client = chat_client
        self._pending: dict[str, tuple[list[Message], list[Message]]] = {}
        self.review_count = 0

    async def _draft(
        self,
        messages: list[Message],
        *,
        revision_feedback: str | None = None,
    ):
        system_text = (
            "Write a concise synthetic operator brief. Include SOURCE and "
            "NEXT ACTION labels. Recommendations are advisory and require "
            "human approval. Do not issue switching commands."
        )
        if revision_feedback:
            system_text += (
                f" Revise the draft using this quality feedback: "
                f"{revision_feedback}"
            )
        return await self._chat_client.get_response(
            messages=[
                Message(role="system", contents=[system_text]),
                *messages,
            ]
        )

    @handler
    async def start(
        self,
        user_messages: list[Message],
        ctx: WorkflowContext[ReviewRequest, AgentResponseUpdate],
    ) -> None:
        response = await self._draft(user_messages)
        request_id = str(uuid4())
        self._pending[request_id] = (user_messages, response.messages)
        await ctx.send_message(
            ReviewRequest(
                request_id=request_id,
                user_messages=user_messages,
                agent_messages=response.messages,
                attempt=1,
            )
        )

    @handler
    async def revise_or_finish(
        self,
        review: ReviewResponse,
        ctx: WorkflowContext[ReviewRequest, AgentResponseUpdate],
    ) -> None:
        self.review_count += 1
        user_messages, last_messages = self._pending[review.request_id]
        if review.approved or review.attempt >= self.MAX_ATTEMPTS:
            contents: list[Content] = []
            for message in last_messages:
                contents.extend(message.contents)
            await ctx.yield_output(
                AgentResponseUpdate(contents=contents, role="assistant")
            )
            self._pending.pop(review.request_id, None)
            return

        revised = await self._draft(
            user_messages,
            revision_feedback=review.feedback,
        )
        self._pending[review.request_id] = (
            user_messages,
            revised.messages,
        )
        await ctx.send_message(
            ReviewRequest(
                request_id=review.request_id,
                user_messages=user_messages,
                agent_messages=revised.messages,
                attempt=review.attempt + 1,
            )
        )

## Participant task

Add one useful required label to the reviewer contract, then update the
generator instructions so it can satisfy the new requirement after feedback.

In [ ]:
# TODO(participant): extend REQUIRED_LABELS with an evidence-related field.
credential = AzureCliCredential()
chat_client = FoundryChatClient(
    project_endpoint=PROJECT_ENDPOINT,
    model=MODEL,
    credential=credential,
)
generator = BriefGenerator(
    id=f"brief-generator-{RESOURCE_NAMESPACE}",
    chat_client=chat_client,
)
reviewer = DeterministicBriefReviewer(
    id=f"brief-reviewer-{RESOURCE_NAMESPACE}"
)
workflow_agent = (
    WorkflowBuilder(start_executor=generator, max_iterations=8)
    .add_edge(generator, reviewer)
    .add_edge(reviewer, generator)
    .build()
    .as_agent(name=f"reviewed-brief-{RESOURCE_NAMESPACE}")
)

In [ ]:
SCENARIO = '''
Synthetic incident INC-1042 reports two high-temperature alarms for TR-104.
Telemetry is available and there is no confirmed customer outage.
Produce a brief for a qualified operator.
'''

streamed: list[str] = []
async for update in workflow_agent.run(SCENARIO, stream=True):
    text = getattr(update, "text", "") or ""
    if text:
        streamed.append(text)
        print(text, end="", flush=True)
final_text = "".join(streamed)

## Deterministic success check

In [ ]:
assert generator.review_count >= 1, "The reviewer did not run."
assert generator.review_count <= generator.MAX_ATTEMPTS
assert final_text.strip(), "The workflow exposed no final brief."
assert RESOURCE_NAMESPACE in generator.id
assert RESOURCE_NAMESPACE in reviewer.id
print(
    f"\nPASS — completed after {generator.review_count} review cycle(s), "
    "within the retry limit."
)

## Optional extension

Replace label checks with a typed response schema and measure first-pass versus
post-revision success on a small evaluation dataset.

**Expected artifact:** a reviewed operator brief and proof that retries are bounded.